In [2]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# 5. Данные пиццерии

Посчитаем p-value для эксперимента с изменением дизайна сайта.

In [3]:
import os
from datetime import datetime, timedelta
import pandas as pd

URL_BASE = 'https://raw.githubusercontent.com/ab-courses/simulator-ab-datasets/main/2022-04-01/'

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

In [4]:
df_sales = read_database('2022-04-01T12_df_sales.csv')
df_sales['date'] = pd.to_datetime(df_sales['date'])
df_users = read_database('experiment_users.csv')

df_sales - информация о покупках, одна строка - один заказ. Атрибуты:
- sale_id - идентификатор покупки;
- date - дата покупки;
- count_pizza - количество пицц в заказе;
- count_drink - количество напитков в заказе;
- price - стоимость заказа;
- user_id - идентификатор пользователя;

df_users - список пользователей эксперимента с разбивкой на группы

In [5]:
df_users.head()

,user_id,pilot
0,0ffc65,0
1,b962b9,0
2,7ea63f,0
3,7f9a61,0
4,459e55,0


In [6]:
# количество пользователей в группах
df_users['pilot'].value_counts()

pilot
0    11769
1    11564
Name: count, dtype: int64

 #### Задание 1
 Проверьте значимость отличий средней выручки с пользователя на неделе перед экспериментом (c 2022.03.16 по 2022.03.23).

Для решения используйте данные из файлов 2022-04-01T12_df_sales.csv и experiment_users.csv.

In [7]:
# считаем значения метрики
begin_date_pred = datetime(2022, 3, 16)
end_date_pred = datetime(2022, 3, 23)
df_metrics_pred = (
    df_sales
    [(df_sales['date'] >= begin_date_pred) & (df_sales['date'] < end_date_pred)]
    .groupby('user_id')[['price']].sum()
    .reset_index()  
)
df_metrics_pred

,user_id,price
0,000096,720
1,00092c,780
2,000bb2,720
3,000ea9,1560
4,000ec6,690
...,...,...
24595,fff3ec,660
24596,fff6b5,3480
24597,fff8bf,2880
24598,fff98d,3480


In [8]:
df_pred = pd.merge(df_users, df_metrics_pred, on='user_id', how='left').fillna(0)

data_control_pred = df_pred[df_pred['pilot'] == 0]['price']

data_pilot_pred = df_pred[df_pred['pilot'] == 1]['price']

stats.ttest_ind(data_control_pred, data_pilot_pred)

TtestResult(statistic=np.float64(-1.2837567415000515), pvalue=np.float64(0.1992398330642495), df=np.float64(23331.0))

**Вывод:** 
- На уровне значимости 0.05 гипотеза о равенстве средних ПРИНИМАЕТСЯ
- Изменение статистически НЕ значимо

 #### Задание 2

Оцените среднее время между покупками. 

Возьмите всех клиентов, у которых 2 и более покупок. Вычислите время между покупками (для клиента с N покупками должно получиться N-1 значения времени). Объедините значения всех клиентов и вычислите среднее.

Для решения используйте данные из файлов 2022-04-01T12_df_sales.csv.

В качестве ответа введите среднее количество дней между покупками, округлённое до целого значения.

In [50]:
df_sales['date'] = pd.to_datetime(df_sales['date'])
# загрузка данных
df = df_sales

# перевод даты в datetime
df['date'] = pd.to_datetime(df['date'])

# сортировка
df = df.sort_values(['user_id', 'date'])

# разница между текущей и следующей покупкой
df['days_between'] = (
    df.groupby('user_id')['date']
      .diff()
      .dt.days
)

df['days_between']

101925     NaN
168536    18.0
90423      NaN
186586    26.0
28831      NaN
          ... 
115269    18.0
51503      NaN
160163     NaN
4685       NaN
37058      8.0
Name: days_between, Length: 203847, dtype: float64

In [52]:
df['days_between'].mean().round()

np.float64(17.0)

#### Задание 3

In [ ]:
import numpy as np
from scipy import stats


def get_ttest_pvalue(metrics_a_group, metrics_b_group):
    #_, pvalue = stats.ttest_ind(metrics_a_group, metrics_b_group)
    return float(stats.ttest_ind(metrics_a_group, metrics_b_group)[1])


metrics_a_group = np.array([964, 1123, 962, 1213, 914, 906, 951, 1033, 987, 1082])
metrics_b_group = np.array([952, 1064, 1091, 1079, 1158, 921, 1161, 1064, 819, 1065])
pvalue = get_ttest_pvalue(metrics_a_group, metrics_b_group)
pvalue
# pvalue = 0.6122191629541949

0.6122191629541951